# Initialization

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor


def evaluate_predictions(y_true_raw, y_pred_log, model_name):
  y_true_log = np.log1p(y_true_raw)

  # 1. RMSLE is equivalent to RMSE in log1p space
  rmsle = root_mean_squared_error(y_true_log, y_pred_log)

  # 2. Invert log transformation and clip negative predictions to 0
  y_pred_raw = np.expm1(y_pred_log)
  y_pred_raw = np.clip(y_pred_raw, 0, None)

  # 3. Calculate metrics on actual visitor counts
  rmse = root_mean_squared_error(y_true_raw, y_pred_raw)
  mae = mean_absolute_error(y_true_raw, y_pred_raw)

  print(f'=== {model_name} ===')
  print(f'RMSLE : {rmsle:.4f}')
  print(f'RMSE  : {rmse:.4f} (visitors)')
  print(f'MAE   : {mae:.4f} (visitors)')

  return {'Model': model_name, 'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae}


# List to store metrics of all 4 models for final comparison
results_summary = []

In [3]:
# Load datasets
train_df = pd.read_csv(r'..\data\restaurant_visits_train.csv')
test_df = pd.read_csv(r'..\data\restaurant_visits_test.csv')

# Define column groups
cat_cols = ['air_genre_name', 'air_area_name']
num_cols = ['month', 'day_of_week_num']
bin_cols = ['holiday_flg', 'is_weekend']
target_col = 'visitors'

# Separate raw targets and log-transformed targets
y_train_raw = train_df[target_col].values
y_test_raw = test_df[target_col].values

y_train_log = np.log1p(y_train_raw)
y_test_log = np.log1p(y_test_raw)

# ---------------------------------------------------------
# PIPELINE A: PREPROCESSING FOR LINEAR REGRESSION
# (Polynomial Degree 2 on numeric + StandardScaler + One-Hot Encoding)
# ---------------------------------------------------------
poly = PolynomialFeatures(degree=2, include_bias=False)
scaler = StandardScaler()
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Fit and transform on Train
X_train_num_poly = scaler.fit_transform(poly.fit_transform(train_df[num_cols]))
X_train_cat_ohe = ohe.fit_transform(train_df[cat_cols])
X_train_bin = train_df[bin_cols].values
X_train_lr = np.hstack([X_train_num_poly, X_train_bin, X_train_cat_ohe])

# Transform on Test
X_test_num_poly = scaler.transform(poly.transform(test_df[num_cols]))
X_test_cat_ohe = ohe.transform(test_df[cat_cols])
X_test_bin = test_df[bin_cols].values
X_test_lr = np.hstack([X_test_num_poly, X_test_bin, X_test_cat_ohe])

# ---------------------------------------------------------
# PIPELINE B: PREPROCESSING FOR CART
# (Target Encoding on categorical using Train log-target mean)
# ---------------------------------------------------------
train_cart = train_df.copy()
test_cart = test_df.copy()
train_cart['target_log'] = y_train_log

target_encode_maps = {}
global_log_mean = y_train_log.mean()

for col in cat_cols:
  mapping = train_cart.groupby(col)['target_log'].mean().to_dict()
  target_encode_maps[col] = mapping
  train_cart[col] = train_cart[col].map(mapping).fillna(global_log_mean)
  test_cart[col] = test_cart[col].map(mapping).fillna(global_log_mean)

cart_feature_cols = cat_cols + num_cols + bin_cols
X_train_cart = train_cart[cart_feature_cols].values
X_test_cart = test_cart[cart_feature_cols].values

print(f'Linear Regression feature matrix shape : {X_train_lr.shape}')
print(f'CART feature matrix shape              : {X_train_cart.shape}')

Linear Regression feature matrix shape : (8004, 28)
CART feature matrix shape              : (8004, 6)


# Algorithm 1: Linear Regression

In [4]:
class CustomRidgeLinearRegression:

  def __init__(self, l2_penalty=0.1):
    self.l2_penalty = l2_penalty
    self.weights = None

  def fit(self, X, y):
    # Add bias column of 1s at index 0
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    n_features = X_b.shape[1]

    # Identity matrix with 0 at (0,0) so we do not penalize the intercept (bias)
    I = np.eye(n_features)
    I[0, 0] = 0.0

    # Closed-form Normal Equation: w = (X^T * X + lambda * I)^(-1) * X^T * y
    A = X_b.T @ X_b + self.l2_penalty * I
    b = X_b.T @ y
    self.weights = np.linalg.solve(A, b)
    return self

  def predict(self, X):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return X_b @ self.weights


# Train and evaluate Custom Linear Regression
lr_scratch = CustomRidgeLinearRegression(l2_penalty=0.1)
lr_scratch.fit(X_train_lr, y_train_log)

y_pred_lr_scratch = lr_scratch.predict(X_test_lr)
results_summary.append(
    evaluate_predictions(
        y_test_raw, y_pred_lr_scratch, '1. Linear Regression (Scratch)'
    )
)

=== 1. Linear Regression (Scratch) ===
RMSLE : 0.7949
RMSE  : 18.4044 (visitors)
MAE   : 12.8159 (visitors)


In [5]:
# Train and evaluate Library Linear Regression (ElasticNetCV tests both L1 and L2)
lr_lib = ElasticNetCV(
    l1_ratio=[0.01, 0.1, 0.5, 0.9, 1.0],
    alphas=[0.001, 0.01, 0.1, 1.0, 10.0],
    cv=5,
    max_iter=2000,
    random_state=42,
)
lr_lib.fit(X_train_lr, y_train_log)

print(
    f'Best alpha: {lr_lib.alpha_:.4f} | Best l1_ratio: {lr_lib.l1_ratio_:.2f}\n'
)

y_pred_lr_lib = lr_lib.predict(X_test_lr)
results_summary.append(
    evaluate_predictions(
        y_test_raw, y_pred_lr_lib, '2. Linear Regression (Library)'
    )
)

Best alpha: 0.0010 | Best l1_ratio: 0.90

=== 2. Linear Regression (Library) ===
RMSLE : 0.7952
RMSE  : 18.4222 (visitors)
MAE   : 12.8230 (visitors)


# Algorithm 2: Decision Tree Regressor (CART)

In [6]:
class TreeNode:

  def __init__(
      self, feature_idx=None, threshold=None, left=None, right=None, value=None
  ):
    self.feature_idx = feature_idx
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value


class CustomCARTRegressor:

  def __init__(self, max_depth=6, min_samples_split=30):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.root = None

  def fit(self, X, y):
    self.root = self._build_tree(X, y, depth=0)
    return self

  def _build_tree(self, X, y, depth):
    n_samples, n_features = X.shape

    # Stopping criteria
    if (
        depth >= self.max_depth
        or n_samples < self.min_samples_split
        or np.var(y) < 1e-7
    ):
      return TreeNode(value=np.mean(y))

    best_feat, best_thresh = self._find_best_split(X, y, n_features)

    # If no valid split reduces error, create a leaf node
    if best_feat is None:
      return TreeNode(value=np.mean(y))

    # Split data and recurse
    left_mask = X[:, best_feat] <= best_thresh
    right_mask = ~left_mask

    left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
    right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

    return TreeNode(
        feature_idx=best_feat,
        threshold=best_thresh,
        left=left_child,
        right=right_child,
    )

  def _find_best_split(self, X, y, n_features):
    best_sse = np.sum((y - np.mean(y)) ** 2)
    best_feat, best_thresh = None, None

    for feat_idx in range(n_features):
      col_values = X[:, feat_idx]
      unique_vals = np.unique(col_values)
      if len(unique_vals) <= 1:
        continue

      # Test midpoints between unique sorted values
      thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0

      for thresh in thresholds:
        left_y = y[col_values <= thresh]
        right_y = y[col_values > thresh]

        # Enforce minimum samples per leaf (half of min_samples_split)
        if (
            len(left_y) < self.min_samples_split // 2
            or len(right_y) < self.min_samples_split // 2
        ):
          continue

        # Calculate weighted Sum of Squared Errors (SSE)
        current_sse = np.sum((left_y - np.mean(left_y)) ** 2) + np.sum(
            (right_y - np.mean(right_y)) ** 2
        )

        if current_sse < best_sse:
          best_sse = current_sse
          best_feat = feat_idx
          best_thresh = thresh

    return best_feat, best_thresh

  def _predict_row(self, row, node):
    if node.value is not None:
      return node.value
    if row[node.feature_idx] <= node.threshold:
      return self._predict_row(row, node.left)
    return self._predict_row(row, node.right)

  def predict(self, X):
    return np.array([self._predict_row(row, self.root) for row in X])


# Train and evaluate Custom CART
cart_scratch = CustomCARTRegressor(max_depth=6, min_samples_split=30)
cart_scratch.fit(X_train_cart, y_train_log)

y_pred_cart_scratch = cart_scratch.predict(X_test_cart)
results_summary.append(
    evaluate_predictions(y_test_raw, y_pred_cart_scratch, '3. CART (Scratch)')
)

=== 3. CART (Scratch) ===
RMSLE : 0.7908
RMSE  : 18.0382 (visitors)
MAE   : 12.6490 (visitors)


In [7]:
# Train and evaluate Library CART
cart_lib = DecisionTreeRegressor(
    max_depth=6, min_samples_split=30, min_samples_leaf=15, random_state=42
)
cart_lib.fit(X_train_cart, y_train_log)

y_pred_cart_lib = cart_lib.predict(X_test_cart)
results_summary.append(
    evaluate_predictions(y_test_raw, y_pred_cart_lib, '4. CART (Library)')
)

=== 4. CART (Library) ===
RMSLE : 0.7908
RMSE  : 18.0382 (visitors)
MAE   : 12.6490 (visitors)


# Comparison & Prediction

In [8]:
# 1. Display overall comparison table
summary_df = pd.DataFrame(results_summary)
print('================ FINAL MODEL COMPARISON ================')
print(summary_df.to_string(index=False))
print('========================================================\n')

# 2. Custom single sample input for testing prediction
sample_input = {
    'air_genre_name': 'Izakaya',  # Restaurant genre
    'air_area_name': 'tokyo',  # Prefecture (lowercase)
    'month': 12,  # December (peak season)
    'day_of_week_num': 5,  # Saturday (0=Mon ... 5=Sat, 6=Sun)
    'holiday_flg': 0,  # Not a national holiday
    'is_weekend': 1,  # Is weekend
}

sample_df = pd.DataFrame([sample_input])
print('Custom Input Sample:')
print(sample_df.to_string(index=False))
print('-' * 56)

# Preprocess sample for Linear Regression models
sample_num_poly = scaler.transform(poly.transform(sample_df[num_cols]))
sample_cat_ohe = ohe.transform(sample_df[cat_cols])
sample_bin = sample_df[bin_cols].values
sample_X_lr = np.hstack([sample_num_poly, sample_bin, sample_cat_ohe])

# Preprocess sample for CART models
sample_cart_df = sample_df.copy()
for col in cat_cols:
  sample_cart_df[col] = (
      sample_cart_df[col]
      .map(target_encode_maps[col])
      .fillna(global_log_mean)
  )
sample_X_cart = sample_cart_df[cart_feature_cols].values

# Predict across all 4 models and convert from log1p back to actual visitor count
pred_lr_scratch = np.expm1(lr_scratch.predict(sample_X_lr)[0])
pred_lr_lib = np.expm1(lr_lib.predict(sample_X_lr)[0])
pred_cart_scratch = np.expm1(cart_scratch.predict(sample_X_cart)[0])
pred_cart_lib = np.expm1(cart_lib.predict(sample_X_cart)[0])

print(
    f'Predicted Visitors - Linear Reg (Scratch) : {pred_lr_scratch:.1f} visitors'
)
print(
    f'Predicted Visitors - Linear Reg (Library) : {pred_lr_lib:.1f} visitors'
)
print(
    f'Predicted Visitors - CART (Scratch)       : {pred_cart_scratch:.1f} visitors'
)
print(
    f'Predicted Visitors - CART (Library)       : {pred_cart_lib:.1f} visitors'
)

================ FINAL MODEL COMPARISON ================
                         Model    RMSLE      RMSE       MAE
1. Linear Regression (Scratch) 0.794916 18.404402 12.815919
2. Linear Regression (Library) 0.795209 18.422162 12.822954
             3. CART (Scratch) 0.790786 18.038205 12.648993
             4. CART (Library) 0.790786 18.038205 12.648993

Custom Input Sample:
air_genre_name air_area_name  month  day_of_week_num  holiday_flg  is_weekend
       Izakaya         tokyo     12                5            0           1
--------------------------------------------------------
Predicted Visitors - Linear Reg (Scratch) : 21.3 visitors
Predicted Visitors - Linear Reg (Library) : 21.0 visitors
Predicted Visitors - CART (Scratch)       : 20.8 visitors
Predicted Visitors - CART (Library)       : 20.8 visitors
